# SMS Spam Detection using Naive Bayes and TF-IDF

This Google Colab notebook trains and evaluates an SMS spam classifier using **TF-IDF** features and **Multinomial Naive Bayes**.

## 1. Upload / load the dataset

The notebook uses the uploaded `spam.csv` file. If running this notebook independently in Colab, upload `spam.csv` when prompted.

In [ ]:
import os
import pandas as pd

CSV_PATH = 'spam.csv'

# In Colab, upload the CSV if it is not already present.
if not os.path.exists(CSV_PATH):
    from google.colab import files
    uploaded = files.upload()
    CSV_PATH = next(iter(uploaded))

df = pd.read_csv(CSV_PATH, encoding='latin-1')
df.head()

In [ ]:
# Prepare the dataset
if 'v1' in df.columns and 'v2' in df.columns:
    data = df[['v1', 'v2']].copy()
    data.columns = ['label', 'message']
else:
    data = df.iloc[:, :2].copy()
    data.columns = ['label', 'message']

data = data.dropna(subset=['label', 'message'])
data['label'] = data['label'].astype(str).str.lower().str.strip()
data['message'] = data['message'].astype(str)

print('Dataset shape:', data.shape)
print('\nClass distribution:')
print(data['label'].value_counts())
data.head()

## 2. Train-test split

In [ ]:
from sklearn.model_selection import train_test_split

X = data['message']
y = data['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print('Training samples:', len(X_train))
print('Testing samples:', len(X_test))

## 3. Convert SMS text into TF-IDF features

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words='english',
    ngram_range=(1, 2),
    max_features=10000
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print('TF-IDF training shape:', X_train_tfidf.shape)
print('TF-IDF testing shape:', X_test_tfidf.shape)

## 4. Train the Multinomial Naive Bayes model

In [ ]:
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()
model.fit(X_train_tfidf, y_train)

print('Model training completed.')

## 5. Evaluate the model

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = model.predict(X_test_tfidf)

print('Accuracy:', round(accuracy_score(y_test, y_pred) * 100, 2), '%')
print('\nClassification Report:')
print(classification_report(y_test, y_pred))

print('Confusion Matrix:')
print(confusion_matrix(y_test, y_pred))

## 6. Test the classifier with your own SMS

In [ ]:
def predict_sms(message):
    message_tfidf = vectorizer.transform([message])
    prediction = model.predict(message_tfidf)[0]
    probability = model.predict_proba(message_tfidf).max()
    return prediction, probability

examples = [
    'Congratulations! You have won a free prize. Call now to claim it.',
    'Hey, are we still meeting at college tomorrow?'
]

for sms in examples:
    prediction, probability = predict_sms(sms)
    print(f'Message: {sms}')
    print(f'Prediction: {prediction.upper()}')
    print(f'Confidence: {probability:.2%}')
    print('-' * 60)

## 7. Save the trained model

The saved files can later be used by a Streamlit application.

In [ ]:
import joblib

joblib.dump(model, 'spam_model.pkl')
joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')

print('Saved: spam_model.pkl')
print('Saved: tfidf_vectorizer.pkl')